# 00 Run Everything - Publication Workflow

This is the single notebook entry point. Run this first for the complete Coswara pipeline and publication extras. The other notebooks are optional review/debug notebooks only.

Default behavior is conservative: expensive or optional stages are controlled by toggles, and existing outputs are skipped unless `FORCE_REBUILD = True`.

In [20]:
from pathlib import Path
import os
import subprocess
import sys
from datetime import datetime

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "covid_audio_btp").exists():
            return candidate
    raise FileNotFoundError("Could not find project root. Start Jupyter from the extracted covid_audio_btp folder or one of its subfolders.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")
print(f"Started: {datetime.now().isoformat(timespec='seconds')}")

Project root: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp
Python: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python
Started: 2026-06-11T18:40:20


## Configuration

Set only the paths and toggles you need. Keep `RUN_CNN = False` until the classical baseline pipeline is clean.

In [21]:
RAW_COSWARA_DIR = PROJECT_ROOT / "data/raw/coswara"
COUGHVID_RAW = None

FORCE_REBUILD = False
RUN_ENV_CHECK = True
RUN_LAYOUT_AUDIT = True

RUN_CORE_COSWARA = True
RUN_FEATURES = True
RUN_ML_BASELINES = True
RUN_CALIBRATION = True
RUN_FUSION = True
RUN_CNN = False
RUN_SHIFT_CHECKS = True
RUN_REPORT_ASSETS = True

RUN_PUBLICATION_EXTRAS = True
RUN_METADATA_BASELINE = True
RUN_QUALITY_WEIGHTED_FUSION = True
RUN_ABSTENTION = True
RUN_BOOTSTRAP_CI = True
RUN_COUGHVID_INDEX = False
RUN_COUGHVID_FEATURES = False
RUN_CROSS_DATASET = False
RUN_PAPER_TABLES = True
RUN_PAIRED_MODEL_COMPARISON = True
RUN_CONFOUNDING_MATCHING = True
RUN_FEATURE_SHIFT_REPORT = False
RUN_EXPERIMENT_MANIFEST = True

MIN_COUGH_DETECTED = 0.80
COUGHVID_FEATURE_MAX_ROWS = 25
CNN_EPOCHS = 50
CNN_BATCH_SIZE = 8

print("Coswara:", RAW_COSWARA_DIR)
print("COUGHVID:", COUGHVID_RAW)

Coswara: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/data/raw/coswara
COUGHVID: None


## Runner

The helper below runs project scripts, checks required inputs, and skips completed outputs unless forced.

In [22]:
def p(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def path_exists(path):
    return p(path).exists() and p(path).stat().st_size > 0


def missing(paths):
    return [str(path) for path in paths if not path_exists(path)]


def run_step(name, args, enabled=True, requires=None, creates=None, strict_requires=True, force=None):
    requires = requires or []
    creates = creates or []
    force = FORCE_REBUILD if force is None else force
    if not enabled:
        print(f"SKIP {name}: disabled")
        return False
    missing_inputs = missing(requires)
    if missing_inputs:
        message = f"SKIP {name}: missing required input(s): {missing_inputs}"
        if strict_requires:
            raise FileNotFoundError(message)
        print(message)
        return False
    if creates and not force and all(path_exists(path) for path in creates):
        print(f"SKIP {name}: outputs already exist")
        return False
    cmd = [str(item) for item in args]
    print()
    print(f"## {name}")
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
    return True


CORE_ARTIFACTS = [
    "data/interim/coswara_index.csv",
    "data/processed/metadata_clean.csv",
    "data/interim/modality_availability.csv",
    "data/interim/split_manifest.csv",
    "data/processed/audio_quality.csv",
    "data/processed/features_mfcc.csv",
    "data/processed/spectrogram_index.csv",
    "data/outputs/metrics/ml_baseline_metrics.csv",
    "data/outputs/metrics/calibrated_branch_predictions.csv",
    "data/outputs/metrics/fusion_metrics.csv",
    "data/outputs/metrics/quality_weighted_fusion_metrics.csv",
    "reports/tables/paper_metric_table.csv",
    "data/outputs/metrics/paired_model_comparison.csv",
    "reports/tables/confounding_balance.csv",
    "reports/tables/feature_shift_report.csv",
    "reports/experiment_manifest.json",
]
print("Runner ready")

Runner ready


## Artifact Dashboard

This shows what already exists before the run.

In [23]:
try:
    import pandas as pd
    dashboard = pd.DataFrame(
        [{"path": path, "exists": path_exists(path), "size_bytes": p(path).stat().st_size if p(path).exists() else 0} for path in CORE_ARTIFACTS]
    )
    display(dashboard)
except Exception as exc:
    print(f"Dashboard unavailable before dependency install: {exc}")
    for path in CORE_ARTIFACTS:
        print(path, "OK" if path_exists(path) else "missing")

,path,exists,size_bytes
0,data/interim/coswara_index.csv,True,14467114
1,data/processed/metadata_clean.csv,True,15192524
2,data/interim/modality_availability.csv,True,211550
3,data/interim/split_manifest.csv,True,243621
4,data/processed/audio_quality.csv,True,8643031
5,data/processed/features_mfcc.csv,True,90512724
6,data/processed/spectrogram_index.csv,True,3879710
7,data/outputs/metrics/ml_baseline_metrics.csv,True,2357
8,data/outputs/metrics/calibrated_branch_predict...,True,1480450
9,data/outputs/metrics/fusion_metrics.csv,True,538


## Stage 0-1: Environment, Index, Splits, Quality

This stage is mandatory before modeling. It prevents training on bad audio, bad labels, or participant leakage.

In [24]:
run_step("local preflight", [sys.executable, "scripts/00_local_preflight.py", "--coswara-dir", RAW_COSWARA_DIR], enabled=True, force=True)

run_step("environment check", [sys.executable, "scripts/00_check_environment.py"], enabled=RUN_ENV_CHECK, force=True)

if RUN_CORE_COSWARA and not RAW_COSWARA_DIR.exists():
    raise FileNotFoundError(f"Coswara not found. Put it at {RAW_COSWARA_DIR} or change RAW_COSWARA_DIR.")

run_step(
    "raw layout audit",
    [sys.executable, "scripts/00_inspect_dataset_layout.py", "--raw-dir", RAW_COSWARA_DIR, "--output", "reports/tables/coswara_layout_audit.csv"],
    enabled=RUN_CORE_COSWARA and RUN_LAYOUT_AUDIT,
    creates=["reports/tables/coswara_layout_audit.csv"],
)
run_step(
    "build Coswara index",
    [sys.executable, "scripts/01_build_coswara_index.py", "--raw-dir", RAW_COSWARA_DIR, "--output", "data/interim/coswara_index.csv"],
    enabled=RUN_CORE_COSWARA,
    creates=["data/interim/coswara_index.csv"],
)
run_step(
    "clean metadata",
    [sys.executable, "scripts/02_clean_metadata.py", "--index", "data/interim/coswara_index.csv", "--output", "data/processed/metadata_clean.csv", "--availability-output", "data/interim/modality_availability.csv", "--audit-output", "reports/tables/dataset_audit.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/coswara_index.csv"],
    creates=["data/processed/metadata_clean.csv", "data/interim/modality_availability.csv"],
)
run_step(
    "participant splits",
    [sys.executable, "scripts/03_create_splits.py", "--metadata", "data/processed/metadata_clean.csv", "--output", "data/interim/split_manifest.csv", "--metadata-output", "data/processed/metadata_clean.csv", "--audit-output", "reports/tables/split_audit.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/interim/split_manifest.csv"],
)
run_step(
    "quality audit",
    [sys.executable, "scripts/04_quality_audit.py", "--metadata", "data/processed/metadata_clean.csv", "--output", "data/processed/audio_quality.csv", "--metadata-output", "data/processed/metadata_clean.csv", "--summary-output", "reports/tables/quality_summary.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/split_manifest.csv", "data/processed/metadata_clean.csv"],
    creates=["data/processed/audio_quality.csv"],
)
run_step(
    "artifact validation",
    [sys.executable, "scripts/12_validate_artifacts.py", "--strict"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/coswara_index.csv", "data/processed/metadata_clean.csv", "data/processed/audio_quality.csv"],
    force=True,
)


## local preflight
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/00_local_preflight.py --coswara-dir /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/data/raw/coswara
Project root: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp
Python: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python
Coswara path: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/data/raw/coswara
Notebook syntax: OK
Python syntax: OK
Required imports: OK
Coswara audio files discovered: 24712
Coswara CSV files discovered: 73

WARNINGS
- xgboost (xgboost): No module named 'xgboost'
- torch (torch): No module named 'torch'
- torchaudio (torchaudio): No module named 'torchaudio'
- streamlit (streamlit): No module named 'streamlit'

Preflight passed. It is safe to start the notebook pipeline.


## environment check
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/00_check_environment.py
Python: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
OK re

True

## Hard Gate

If this cell fails, inspect the optional review notebooks before training models.

In [25]:
import pandas as pd

from covid_audio_btp.notebook_utils import (
    assert_binary_labels_present,
    assert_no_participant_leakage,
    check_artifacts,
    read_optional_csv,
    stop_if_validation_errors,
)

metadata = pd.read_csv(p("data/processed/metadata_clean.csv"))
quality = pd.read_csv(p("data/processed/audio_quality.csv"))
issues = read_optional_csv("reports/tables/validation_issues.csv")

assert_binary_labels_present(metadata)
assert_no_participant_leakage(metadata)
if issues is not None:
    stop_if_validation_errors(issues)

quality_ok_rate = float((quality["quality_flag"] == "ok").mean()) if len(quality) else 0.0
print(f"quality ok rate: {quality_ok_rate:.3f}")
if quality_ok_rate < 0.50:
    raise AssertionError("Quality ok rate below 50%; review audio quality before modeling.")

reports/tables/validation_issues.csv: 1 rows x 3 columns
label gate passed: both positive and negative classes are present
leakage gate passed: 2746 participants appear in one split each
validation gate passed with warnings only
quality ok rate: 0.953


## Stage 2-5: Features, Baselines, Calibration, Fusion

This is the main Coswara modeling path. CNN is optional and disabled by default.

In [26]:
run_step(
    "feature and spectrogram extraction",
    [sys.executable, "scripts/05_extract_features.py", "--metadata", "data/processed/metadata_clean.csv", "--features-output", "data/processed/features_mfcc.csv", "--spectrogram-dir", "data/processed/spectrograms", "--spectrogram-index-output", "data/processed/spectrogram_index.csv", "--quality-mode", "all_samples"],
    enabled=RUN_FEATURES,
    requires=["data/processed/metadata_clean.csv", "data/processed/audio_quality.csv"],
    creates=["data/processed/features_mfcc.csv", "data/processed/spectrogram_index.csv"],
)
run_step(
    "classical ML baselines",
    [sys.executable, "scripts/06_train_ml_baselines.py", "--features", "data/processed/features_mfcc.csv"],
    enabled=RUN_ML_BASELINES,
    requires=["data/processed/features_mfcc.csv"],
    creates=["data/outputs/metrics/ml_baseline_metrics.csv", "data/outputs/metrics/ml_validation_metrics.csv", "data/outputs/metrics/ml_predictions_validation.csv", "data/outputs/metrics/ml_predictions_test.csv"],
)
run_step(
    "compact CNN cough branch",
    [sys.executable, "scripts/07_train_cnn.py", "--spectrogram-index", "data/processed/spectrogram_index.csv", "--modality", "cough", "--epochs", CNN_EPOCHS, "--batch-size", CNN_BATCH_SIZE],
    enabled=RUN_CNN,
    requires=["data/processed/spectrogram_index.csv"],
    creates=["data/outputs/metrics/cnn_metrics.csv", "data/outputs/metrics/cnn_logits_validation.csv", "data/outputs/metrics/cnn_logits_test.csv"],
)
run_step(
    "branch calibration",
    [sys.executable, "scripts/08_calibrate_branches.py", "--validation-predictions", "data/outputs/metrics/ml_predictions_validation.csv", "--test-predictions", "data/outputs/metrics/ml_predictions_test.csv", "--method", "platt"],
    enabled=RUN_CALIBRATION,
    requires=["data/outputs/metrics/ml_predictions_validation.csv", "data/outputs/metrics/ml_predictions_test.csv"],
    creates=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/outputs/metrics/calibration_metrics.csv"],
)
run_step(
    "standard fusion",
    [sys.executable, "scripts/09_run_fusion.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--validation-metrics", "data/outputs/metrics/ml_validation_metrics.csv"],
    enabled=RUN_FUSION,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/outputs/metrics/ml_validation_metrics.csv"],
    creates=["data/outputs/metrics/fusion_predictions.csv", "data/outputs/metrics/fusion_metrics.csv"],
)
run_step(
    "subgroup and confounding checks",
    [sys.executable, "scripts/10_shift_and_confounding_checks.py", "--predictions", "data/outputs/metrics/fusion_predictions.csv", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_SHIFT_CHECKS,
    requires=["data/outputs/metrics/fusion_predictions.csv", "data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/subgroup_metrics.csv"],
)
run_step(
    "basic report assets",
    [sys.executable, "scripts/11_make_report_assets.py", "--metadata", "data/processed/metadata_clean.csv", "--predictions", "data/outputs/metrics/fusion_predictions.csv"],
    enabled=RUN_REPORT_ASSETS,
    requires=["data/processed/metadata_clean.csv"],
    creates=["reports/report_outline.md", "reports/slides_outline.md"],
)

SKIP feature and spectrogram extraction: outputs already exist
SKIP classical ML baselines: outputs already exist
SKIP compact CNN cough branch: disabled
SKIP branch calibration: outputs already exist
SKIP standard fusion: outputs already exist
SKIP subgroup and confounding checks: outputs already exist
SKIP basic report assets: outputs already exist


False

## Stage 6: Publication Extras

These experiments support the publication-grade claims: metadata-only baseline, quality-weighted fusion, abstention, confidence intervals, COUGHVID external validation, and paper tables.

In [27]:
run_step(
    "metadata-only baseline",
    [sys.executable, "scripts/14_train_metadata_baseline.py", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_METADATA_BASELINE,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/metadata_baseline_metrics.csv", "data/outputs/metrics/metadata_predictions_validation.csv", "data/outputs/metrics/metadata_predictions_test.csv"],
)
run_step(
    "quality-weighted fusion",
    [sys.executable, "scripts/16_run_quality_weighted_fusion.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--quality", "data/processed/audio_quality.csv", "--validation-metrics", "data/outputs/metrics/ml_validation_metrics.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_QUALITY_WEIGHTED_FUSION,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/processed/audio_quality.csv", "data/outputs/metrics/ml_validation_metrics.csv"],
    creates=["data/outputs/metrics/quality_weighted_fusion_predictions.csv", "data/outputs/metrics/quality_weighted_fusion_metrics.csv"],
)
run_step(
    "abstention analysis",
    [sys.executable, "scripts/17_abstention_analysis.py", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_ABSTENTION,
    requires=["data/outputs/metrics/quality_weighted_fusion_predictions.csv", "data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/abstention_decisions.csv", "data/outputs/metrics/abstention_coverage_curve.csv"],
)
run_step(
    "bootstrap CI for quality-weighted fusion",
    [sys.executable, "scripts/15_bootstrap_prediction_metrics.py", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--output", "data/outputs/metrics/quality_weighted_fusion_bootstrap_ci.csv", "--group-columns", "fusion_method"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_BOOTSTRAP_CI,
    requires=["data/outputs/metrics/quality_weighted_fusion_predictions.csv"],
    creates=["data/outputs/metrics/quality_weighted_fusion_bootstrap_ci.csv"],
)

if COUGHVID_RAW is not None:
    run_step(
        "COUGHVID index",
        [sys.executable, "scripts/13_build_coughvid_index.py", "--raw-dir", COUGHVID_RAW, "--output", "data/interim/coughvid_index.csv", "--min-cough-detected", MIN_COUGH_DETECTED],
        enabled=RUN_PUBLICATION_EXTRAS and RUN_COUGHVID_INDEX,
        creates=["data/interim/coughvid_index.csv"],
    )
else:
    print("SKIP COUGHVID index: COUGHVID_RAW is None")

feature_cmd = [sys.executable, "scripts/19_extract_coughvid_features.py", "--index", "data/interim/coughvid_index.csv", "--features-output", "data/processed/coughvid_features_mfcc.csv", "--quality-ok-only"]
if COUGHVID_FEATURE_MAX_ROWS is not None:
    feature_cmd += ["--max-rows", COUGHVID_FEATURE_MAX_ROWS]
run_step(
    "COUGHVID feature extraction",
    feature_cmd,
    enabled=RUN_PUBLICATION_EXTRAS and RUN_COUGHVID_FEATURES,
    requires=["data/interim/coughvid_index.csv"],
    creates=["data/processed/coughvid_features_mfcc.csv"],
    strict_requires=False,
)
run_step(
    "cross-dataset cough evaluation",
    [sys.executable, "scripts/18_cross_dataset_feature_eval.py", "--source-features", "data/processed/features_mfcc.csv", "--external-features", "data/processed/coughvid_features_mfcc.csv", "--modality", "cough", "--model-name", "logistic_regression"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_CROSS_DATASET,
    requires=["data/processed/features_mfcc.csv", "data/processed/coughvid_features_mfcc.csv"],
    creates=["data/outputs/metrics/cross_dataset_predictions.csv", "data/outputs/metrics/cross_dataset_metrics.csv"],
    strict_requires=False,
)
run_step(
    "paper metric tables",
    [sys.executable, "scripts/20_make_paper_tables.py"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_PAPER_TABLES,
    requires=["data/outputs/metrics/ml_baseline_metrics.csv"],
    creates=["reports/tables/paper_metric_table.csv", "reports/tables/paper_metric_table_raw.csv"],
    strict_requires=False,
)

SKIP metadata-only baseline: outputs already exist

## quality-weighted fusion
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/16_run_quality_weighted_fusion.py --predictions data/outputs/metrics/calibrated_branch_predictions.csv --quality data/processed/audio_quality.csv --validation-metrics data/outputs/metrics/ml_validation_metrics.csv
Wrote quality-weighted fusion thresholds: data/outputs/metrics/quality_weighted_fusion_thresholds.csv
Wrote quality-weighted fusion predictions: data/outputs/metrics/quality_weighted_fusion_predictions.csv
Wrote quality-weighted fusion metrics: data/outputs/metrics/quality_weighted_fusion_metrics.csv


## abstention analysis
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/17_abstention_analysis.py --predictions data/outputs/metrics/quality_weighted_fusion_predictions.csv --metadata data/processed/metadata_clean.csv
Wrote abstention decisions: data/outputs/metrics/abstention_decisions.csv
Wrote ab

True

## Stage 7: Extra Publication Strengthening

These optional diagnostics make the paper stronger: paired model comparison, matched confounding subset, source-vs-external feature shift, and a reproducibility manifest.

In [28]:
run_step(
    "paired ML model comparison",
    [sys.executable, "scripts/21_paired_model_comparison.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--baseline-name", "logistic_regression", "--model-column", "model_name", "--group-columns", "modality", "--output", "data/outputs/metrics/paired_model_comparison.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_PAIRED_MODEL_COMPARISON,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv"],
    creates=["data/outputs/metrics/paired_model_comparison.csv"],
    strict_requires=False,
)
run_step(
    "confounding matched subset",
    [sys.executable, "scripts/22_confounding_matching.py", "--metadata", "data/processed/metadata_clean.csv", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--covariates", "age_bucket", "gender"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_CONFOUNDING_MATCHING,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/processed/metadata_matched.csv", "reports/tables/confounding_balance.csv"],
    strict_requires=False,
)
run_step(
    "feature shift report",
    [sys.executable, "scripts/23_feature_shift_report.py", "--source-features", "data/processed/features_mfcc.csv", "--external-features", "data/processed/coughvid_features_mfcc.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_FEATURE_SHIFT_REPORT,
    requires=["data/processed/features_mfcc.csv", "data/processed/coughvid_features_mfcc.csv"],
    creates=["reports/tables/feature_shift_report.csv", "reports/tables/feature_shift_summary.csv"],
    strict_requires=False,
)
run_step(
    "experiment manifest",
    [sys.executable, "scripts/24_make_experiment_manifest.py", "--run-name", "covid_audio_publication_run"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_EXPERIMENT_MANIFEST,
    creates=["reports/experiment_manifest.json"],
)

SKIP paired ML model comparison: outputs already exist

## confounding matched subset
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/22_confounding_matching.py --metadata data/processed/metadata_clean.csv --predictions data/outputs/metrics/quality_weighted_fusion_predictions.csv --covariates age_bucket gender
Wrote matched metadata: data/processed/metadata_matched.csv (11196 rows)
Wrote balance table: reports/tables/confounding_balance.csv
Wrote matched-subset metrics: data/outputs/metrics/matched_subset_metrics.csv

SKIP feature shift report: disabled

## experiment manifest
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/24_make_experiment_manifest.py --run-name covid_audio_publication_run
Wrote experiment manifest: reports/experiment_manifest.json



True

## Final Dashboard

Use this at the end of the run to see exactly what was produced.

In [29]:
import pandas as pd

final_dashboard = pd.DataFrame(
    [{"path": path, "exists": path_exists(path), "size_bytes": p(path).stat().st_size if p(path).exists() else 0} for path in CORE_ARTIFACTS]
)
display(final_dashboard)

for path in [
    "data/outputs/metrics/ml_baseline_metrics.csv",
    "data/outputs/metrics/fusion_metrics.csv",
    "data/outputs/metrics/quality_weighted_fusion_metrics.csv",
    "reports/tables/paper_metric_table.csv",
    "data/outputs/metrics/paired_model_comparison.csv",
    "reports/tables/confounding_balance.csv",
    "reports/tables/feature_shift_report.csv",
    "reports/experiment_manifest.json",
]:
    if path_exists(path):
        print()
        print(path)
        display(pd.read_csv(p(path)).head(20))

print("Run finished")

,path,exists,size_bytes
0,data/interim/coswara_index.csv,True,14467114
1,data/processed/metadata_clean.csv,True,15192524
2,data/interim/modality_availability.csv,True,211550
3,data/interim/split_manifest.csv,True,243621
4,data/processed/audio_quality.csv,True,8643031
5,data/processed/features_mfcc.csv,True,90512724
6,data/processed/spectrogram_index.csv,True,3879710
7,data/outputs/metrics/ml_baseline_metrics.csv,True,2357
8,data/outputs/metrics/calibrated_branch_predict...,True,1480450
9,data/outputs/metrics/fusion_metrics.csv,True,538



data/outputs/metrics/ml_baseline_metrics.csv


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,brier,ece,nll,threshold,n_samples,model_name,modality
0,0.500000,0.323899,0.500000,0.000000,0.000000,1.000000,0.323899,0.323899,4.474836,0.5,636.0,dummy_most_frequent,cough
1,0.479454,0.315672,0.479454,0.296117,0.296117,0.662791,0.455975,0.455975,6.299526,0.5,636.0,dummy_stratified,cough
2,0.788530,0.654073,0.738451,0.646586,0.781553,0.695349,0.194444,0.162217,0.583684,0.5,636.0,logistic_regression,cough
3,0.808015,0.720197,0.659844,0.498316,0.359223,0.960465,0.170637,0.132054,0.522154,0.5,636.0,random_forest,cough
4,0.500000,0.323899,0.500000,0.000000,0.000000,1.000000,0.323899,0.323899,4.474836,0.5,636.0,dummy_most_frequent,breath
5,0.479454,0.315672,0.479454,0.296117,0.296117,0.662791,0.455975,0.455975,6.299526,0.5,636.0,dummy_stratified,breath
6,0.755791,0.567941,0.696286,0.601905,0.766990,0.625581,0.205722,0.146322,0.595590,0.5,636.0,logistic_regression,breath
7,0.792527,0.664178,0.684929,0.555891,0.446602,0.923256,0.170282,0.074652,0.519347,0.5,636.0,random_forest,breath
8,0.500000,0.323899,0.500000,0.000000,0.000000,1.000000,0.323899,0.323899,4.474836,0.5,1590.0,dummy_most_frequent,speech
9,0.514595,0.330683,0.514595,0.342746,0.341748,0.687442,0.424528,0.424528,5.865076,0.5,1590.0,dummy_stratified,speech



data/outputs/metrics/fusion_metrics.csv


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,brier,ece,nll,threshold,n_samples,fusion_method
0,0.878709,0.840627,0.810996,0.743961,0.747573,0.874419,0.189878,0.152085,0.565095,0.346350,318.0,uniform_mean
1,0.877670,0.842766,0.812305,0.753927,0.699029,0.925581,0.189645,0.143924,0.564572,0.357049,318.0,validation_weighted_auprc


...


reports/tables/confounding_balance.csv


,covariate,type,n_positive,n_negative,max_abs_standardized_difference,details_json
0,age_bucket,categorical,5598,5598,0.0,"{""30-44"":0.0,""45-59"":0.0,""60+"":0.0,""<30"":0.0}"
1,gender,categorical,5598,5598,0.0,"{""female"":0.0,""male"":0.0,""other"":0.0}"



reports/experiment_manifest.json


,{
"""created_at_utc"": ""2026-06-11T13:10:30.987100+00:00""",NaN
"""platform"": ""Linux-6.17.0-35-generic-x86_64-with-glibc2.39""",NaN
"""python_executable"": ""/home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python""",NaN
"""config"": {",NaN
"""run_name"": ""covid_audio_publication_run""",NaN
"""seed"": 42",NaN
},NaN
"""packages"": {",NaN
"""python"": ""3.12.3""",NaN
"""numpy"": ""2.4.6""",NaN


Run finished


In [30]:
from pathlib import Path
from IPython.display import display
import subprocess
import sys
import pandas as pd


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python kernel:", sys.executable)

if ".venv" not in sys.executable:
    raise RuntimeError("Wrong kernel. In Jupyter, select the .venv kernel first.")


required = [
    "data/outputs/metrics/cnn_metrics.csv",
    "data/outputs/metrics/cnn_logits_validation.csv",
    "data/outputs/metrics/cnn_logits_test.csv",
    "data/outputs/metrics/cnn_training_history.csv",
    "data/outputs/models/compact_cnn_cough.pt",
]

missing = [p for p in required if not (PROJECT_ROOT / p).exists()]
if missing:
    raise FileNotFoundError(f"Missing CNN outputs: {missing}")

print("\nCNN metrics:")
cnn_metrics = pd.read_csv(PROJECT_ROOT / "data/outputs/metrics/cnn_metrics.csv")
display(cnn_metrics)

print("\nCNN training history:")
cnn_history = pd.read_csv(PROJECT_ROOT / "data/outputs/metrics/cnn_training_history.csv")
print("epochs actually run:", int(cnn_history["epoch"].max()))
display(cnn_history.tail(15))


def run(cmd):
    print("\n$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()


run([sys.executable, "scripts/20_make_paper_tables.py"])
run([sys.executable, "scripts/24_make_experiment_manifest.py"])
run([sys.executable, "scripts/12_validate_artifacts.py", "--strict"])

paper = pd.read_csv(PROJECT_ROOT / "reports/tables/paper_metric_table.csv")

print("\nCNN row inside final paper table:")
display(paper[paper["table_source"].eq("cnn_metrics")])

print("\nTop audio-model rows by AUPRC:")
audio_sources = [
    "ml_baseline_metrics",
    "cnn_metrics",
    "fusion_metrics",
    "quality_weighted_fusion_metrics",
]
audio_rows = paper[paper["table_source"].isin(audio_sources)].copy()
display(audio_rows.sort_values("auprc", ascending=False).head(15))

print("\nDone.")

PROJECT_ROOT: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp
Python kernel: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python

CNN metrics:


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,brier,ece,nll,threshold,n_samples,model_name,modality
0,0.749853,0.558663,0.668785,0.57754,0.786408,0.551163,0.223665,0.196838,0.642371,0.5,636.0,compact_cnn,cough



CNN training history:
epochs actually run: 22


,epoch,train_loss,validation_loss
7,8,0.880609,0.977786
8,9,0.875669,0.846615
9,10,0.856301,0.870694
10,11,0.864517,0.852421
11,12,0.864658,0.830813
12,13,0.851376,0.834436
13,14,0.852004,0.818315
14,15,0.838088,0.959348
15,16,0.843144,0.846782
16,17,0.836331,0.824188



$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/20_make_paper_tables.py
Wrote paper metric table: reports/tables/paper_metric_table.csv (29 rows)
Wrote raw combined metric table: reports/tables/paper_metric_table_raw.csv (29 rows)


$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/24_make_experiment_manifest.py
Wrote experiment manifest: reports/experiment_manifest.json


$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/12_validate_artifacts.py --strict
severity          check                       message
 warning unknown_labels 5688 rows have unknown labels


CNN row inside final paper table:


,table_source,model_name,modality,fusion_method,n_samples,auroc,auprc,balanced_accuracy,sensitivity,specificity,f1,brier,ece,nll
12,cnn_metrics,compact_cnn,cough,NaN,636,0.750,0.559,0.669,0.786,0.551,0.578,0.224,0.197,0.642



Top audio-model rows by AUPRC:


,table_source,model_name,modality,fusion_method,n_samples,auroc,auprc,balanced_accuracy,sensitivity,specificity,f1,brier,ece,nll
26,fusion_metrics,NaN,NaN,validation_weighted_auprc,318,0.878,0.843,0.812,0.699,0.926,0.754,0.190,0.144,0.565
25,fusion_metrics,NaN,NaN,uniform_mean,318,0.879,0.841,0.811,0.748,0.874,0.744,0.190,0.152,0.565
27,quality_weighted_fusion_metrics,NaN,NaN,quality_weighted_auprc,318,"0.879 [0.831, 0.921]","0.838 [0.775, 0.888]",0.791,0.816,0.767,0.709,"0.190 [0.174, 0.205]","0.157 [0.130, 0.203]",0.565
3,ml_baseline_metrics,random_forest,cough,NaN,636,0.808,0.720,0.660,0.359,0.960,0.498,0.171,0.132,0.522
7,ml_baseline_metrics,random_forest,breath,NaN,636,0.793,0.664,0.685,0.447,0.923,0.556,0.170,0.075,0.519
2,ml_baseline_metrics,logistic_regression,cough,NaN,636,0.789,0.654,0.738,0.782,0.695,0.647,0.194,0.162,0.584
11,ml_baseline_metrics,random_forest,speech,NaN,1590,0.758,0.621,0.635,0.340,0.931,0.458,0.181,0.075,0.544
10,ml_baseline_metrics,logistic_regression,speech,NaN,1590,0.749,0.605,0.674,0.734,0.613,0.578,0.210,0.162,0.604
6,ml_baseline_metrics,logistic_regression,breath,NaN,636,0.756,0.568,0.696,0.767,0.626,0.602,0.206,0.146,0.596
12,cnn_metrics,compact_cnn,cough,NaN,636,0.750,0.559,0.669,0.786,0.551,0.578,0.224,0.197,0.642



Done.
